# 🚀 M.Tech PEFT-SLM — Full Pipeline on Google Colab
**Parameter-Efficient Fine-Tuning of Small Language Models for Document Analysis**

---
### How to use this notebook
Run every cell **top to bottom** in order. Each section has a ✅ checkpoint — check it before moving on.

| Section | What it does | GPU needed? | Time |
|---|---|---|---|
| 0 — Setup | Clone repo, install packages | ❌ CPU | ~5 min |
| 1 — GPU Check | Verify T4 is available | ✅ GPU | <1 min |
| 2 — Smoke Test | Verify every pipeline component | ✅ GPU | ~5 min |
| 3 — Quick Demo | 3-way comparison on 5 examples | ✅ GPU | ~10 min |
| 4 — Data Prep | Download & process SQuAD + XL-Sum | ❌ CPU | ~5 min |
| 5A — SQuAD QA | Base SLM baseline | ✅ GPU | ~20 min |
| 5B — SQuAD QA | LoRA fine-tuning ← your result | ✅ GPU | ~90 min |
| 5C — SQuAD QA | 7B LLM baseline | ✅ GPU | ~25 min |
| 5D — SQuAD QA | Compile 3-way table | ✅ GPU | <1 min |
| 6A — XL-Sum | Base SLM baseline | ✅ GPU | ~20 min |
| 6B — XL-Sum | LoRA fine-tuning | ✅ GPU | ~2 hr |
| 7 — Results | Print + plot all results | ❌ CPU | <1 min |

> **First time?** Go to `Runtime → Change runtime type → T4 GPU` before running anything.


---
## 🔧 Section 0 — Setup: Clone Repo & Install Dependencies

In [ ]:
# ── 0.1  Clone your project from GitHub ───────────────────────────────────────
# Replace the URL below with YOUR repository URL.
# If you don't have a repo yet, upload your project folder to Google Drive instead
# and run the Google Drive mount cell below.

import os

REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"   # ← change this
PROJECT_DIR = "/content/Mtech"

USE_GITHUB = False   # Set True if you have a GitHub repo
USE_GDRIVE = True    # Set True to mount Google Drive

if USE_GITHUB:
    if not os.path.exists(PROJECT_DIR):
        !git clone {REPO_URL} {PROJECT_DIR}
    else:
        print("Repo already cloned — pulling latest changes...")
        !git -C {PROJECT_DIR} pull

elif USE_GDRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    # Adjust the path to where your project is stored in Drive:
    PROJECT_DIR = "/content/drive/MyDrive/Mtech"   # ← change if needed
    print(f"Using project at: {PROJECT_DIR}")

else:
    # Fallback: project already in /content (e.g. uploaded via Files panel)
    PROJECT_DIR = "/content/Mtech"

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")
!ls

In [ ]:
# ── 0.2  Install all dependencies ─────────────────────────────────────────────
# This takes ~3-5 minutes on first run. Subsequent runs are faster.

!pip install -q -r requirements.txt
print("\n✅ Dependencies installed")

In [ ]:
# ── 0.3  Verify key imports ────────────────────────────────────────────────────
import importlib, sys

packages = {
    "torch": "torch",
    "transformers": "transformers",
    "peft": "peft",
    "datasets": "datasets",
    "evaluate": "evaluate",
    "accelerate": "accelerate",
    "bitsandbytes": "bitsandbytes",
}

all_ok = True
for name, mod in packages.items():
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, "__version__", "?")
        print(f"  [OK] {name:20s} {ver}")
    except ImportError as e:
        print(f"  [FAIL] {name}: {e}")
        all_ok = False

if all_ok:
    print("\n✅ All imports OK — ready to continue")
else:
    print("\n❌ Some imports failed — re-run the install cell above")

In [ ]:
# ── 0.4  Run the project validation script ────────────────────────────────────
!python validate_setup.py

---
## 🖥️ Section 1 — GPU Check
> ✅ Checkpoint: You should see your GPU name and ≥ 6 GB VRAM.

In [ ]:
import torch

if not torch.cuda.is_available():
    print("❌  No GPU detected!")
    print("   Go to: Runtime → Change runtime type → Hardware accelerator → T4 GPU")
    print("   Then: Runtime → Disconnect and delete runtime, then reconnect.")
else:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅  GPU   : {gpu_name}")
    print(f"✅  VRAM  : {vram_gb:.1f} GB")
    print(f"   PyTorch: {torch.__version__}")
    print(f"   CUDA   : {torch.version.cuda}")

    if vram_gb < 6:
        print("⚠️  Less than 6 GB VRAM — add `load_in_bits: 4` to YAML configs (QLoRA mode)")
    elif vram_gb < 12:
        print("ℹ️  T4-class GPU — recommend load_in_bits: 4 for the 7B baseline")
    else:
        print("👍  Plenty of VRAM — no quantization needed")

---
## 🧪 Section 2 — Smoke Test (9 checks)
This verifies every component of the pipeline works correctly on THIS machine.

> ✅ Checkpoint: All 9 checks must print `[OK]` and the final line must say `ALL SMOKE TEST CHECKS PASSED`

In [ ]:
!python smoke_test.py

---
## ⚡ Section 3 — Quick 3-Way Demo (5 built-in examples)
This runs the full pipeline on 5 hard-coded examples in ~10 minutes — no dataset download needed.
It shows you the comparison table format you'll get from the full experiments.

> ✅ Checkpoint: A 3-way comparison table should print at the end with EM and F1 values.

In [ ]:
# ── 3.1  QA demo (skip LLM to save time on first run) ─────────────────────────
!python demo_quick.py --task qa --lora_steps 10 --skip_llm

In [ ]:
# ── 3.2  Summarization demo ────────────────────────────────────────────────────
!python demo_quick.py --task summarization --lora_steps 10 --skip_llm

---
## 📦 Section 4 — Prepare Datasets
Downloads SQuAD v2 and XL-Sum from Hugging Face Hub and applies prompt transformations.
Only run this if `data/processed/` does not already exist (it was prepared on your local machine).

> ✅ Checkpoint: `data/processed/squad_v2/train.jsonl` and `data/processed/xlsum_en/train.jsonl` must exist.

In [ ]:
import os

squad_ok = os.path.exists("data/processed/squad_v2/train.jsonl")
xlsum_ok  = os.path.exists("data/processed/xlsum_en/train.jsonl")

print(f"SQuAD v2 data : {'✅ exists' if squad_ok else '❌ missing — run cell below'}")
print(f"XL-Sum data   : {'✅ exists' if xlsum_ok else '❌ missing — run cell below'}")

In [ ]:
# ── 4.1  Prepare SQuAD v2 (skip if already exists) ────────────────────────────
import os
if not os.path.exists("data/processed/squad_v2/train.jsonl"):
    print("Preparing SQuAD v2...")
    !python scripts/prepare_data.py --config configs/pilot_squad_qa.yaml
else:
    print("✅ SQuAD v2 already prepared — skipping")

if not os.path.exists("data/processed/xlsum_en/train.jsonl"):
    print("\nPreparing XL-Sum...")
    !python scripts/prepare_data.py --config configs/pilot_xlsum_summarization.yaml
else:
    print("✅ XL-Sum already prepared — skipping")

In [ ]:
# ── 4.2  Verify split sizes ────────────────────────────────────────────────────
import json, pathlib

def count_lines(path):
    p = pathlib.Path(path)
    return sum(1 for _ in p.open()) if p.exists() else 0

datasets_info = [
    ("SQuAD v2", "data/processed/squad_v2"),
    ("XL-Sum EN", "data/processed/xlsum_en"),
]

for name, folder in datasets_info:
    tr = count_lines(f"{folder}/train.jsonl")
    va = count_lines(f"{folder}/validation.jsonl")
    te = count_lines(f"{folder}/test.jsonl")
    status = "✅" if tr > 0 and va > 0 and te > 0 else "❌"
    print(f"{status} {name:12s} | train={tr:>5}  val={va:>4}  test={te:>4}")

print("\nExpected: train=5000  val=500  test=500 for both datasets")

In [ ]:
# ── 4.3  Run smoke_test.py again to confirm everything is clean ────────────────
!python smoke_test.py

---
## 📊 Section 5 — SQuAD v2 QA Experiments (Phase 1B)
This is the core of Phase 1. It produces the 3-way comparison table for your report.

Run steps 5A → 5B → 5C → 5D in order.

In [ ]:
# ── 5A  Base SLM zero-shot baseline (~15-20 min) ───────────────────────────────
# This is Row 1 of your results table.
# Expected: low EM/F1 — the base model is not instruction-tuned.

print("Running Base SLM zero-shot baseline on SQuAD v2...")
print("Expected time: ~15-20 minutes on T4\n")

!python src/training/run_pilot.py \
    --config configs/pilot_squad_qa.yaml \
    --base_only

print("\n✅ Step 5A complete — results logged to experiments/results.csv")

In [ ]:
# ── 5B  LoRA fine-tuning — YOUR RESEARCH CONTRIBUTION (~90 min) ───────────────
# This trains the PEFT-adapted SLM and evaluates it on the same 500 test examples.
# Watch for: loss decreasing step by step.
# If you get OOM error: add `load_in_bits: 4` in configs/pilot_squad_qa.yaml

print("Training SLM + LoRA adapter (PEFT) on SQuAD v2...")
print("Expected time: ~90 min on T4 (3 epochs × 5000 examples)")
print("Watch: loss should decrease. Best checkpoint saved to experiments/squad_qa/adapter/\n")

!python src/training/run_pilot.py \
    --config configs/pilot_squad_qa.yaml

print("\n✅ Step 5B complete — LoRA adapter saved, results logged")

In [ ]:
# ── 5B (OOM fix) — Only run this if you got an OOM error above ────────────────
# This patches the config to use 4-bit QLoRA (uses ~half the VRAM)

import yaml, pathlib

cfg_path = pathlib.Path("configs/pilot_squad_qa.yaml")
cfg = yaml.safe_load(cfg_path.read_text())

cfg["load_in_bits"] = 4
cfg_path.write_text(yaml.dump(cfg, default_flow_style=False))

print("✅ Set load_in_bits: 4 (QLoRA mode) — re-run the 5B training cell now")

In [ ]:
# ── 5C  LLM baseline — Qwen1.5-7B zero-shot (~20-30 min) ──────────────────────
# This is Row 3 of your results table.
# If OOM: edit configs/llm_baseline_qwen7b.yaml and set load_in_bits: 4

print("Running LLM baseline (Qwen1.5-7B zero-shot) on SQuAD v2...")
print("Expected time: ~20-30 min on T4")
print("If OOM: run the OOM-fix cell below\n")

!python src/training/run_pilot.py \
    --config configs/llm_baseline_qwen7b.yaml \
    --base_only

print("\n✅ Step 5C complete — LLM baseline results logged")

In [ ]:
# ── 5C (OOM fix) — Only run if 7B model gives OOM ─────────────────────────────
import yaml, pathlib

cfg_path = pathlib.Path("configs/llm_baseline_qwen7b.yaml")
cfg = yaml.safe_load(cfg_path.read_text())
cfg["load_in_bits"] = 4
cfg_path.write_text(yaml.dump(cfg, default_flow_style=False))

print("✅ Set load_in_bits: 4 for LLM config — re-run 5C cell")

In [ ]:
# ── 5D  Compile the 3-way QA comparison table ─────────────────────────────────

from src.utils.experiment_tracker import ExperimentTracker

print("=" * 70)
print("3-WAY COMPARISON — SQuAD v2 QA — 500 test examples — seed=42")
print("=" * 70)
ExperimentTracker("experiments/results.csv").print_summary()
print("\n✅ Results saved to experiments/results.csv")

In [ ]:
# ── 5D (extra) — Print the LoRA parameter report ──────────────────────────────
# This shows the trainable % to use in your report (replaces the "~0.4%" estimate)

import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen1.5-0.5B",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
peft_model = PeftModel.from_pretrained(base, "experiments/squad_qa/adapter")

total     = sum(p.numel() for p in peft_model.parameters())
trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
pct = 100 * trainable / total

print("┌─────────────────────────────────────────────────────────────┐")
print("│  LoRA Parameter Report                                      │")
print(f"│  Total model parameters    : {total:>12,}              │")
print(f"│  Trainable (LoRA only)     : {trainable:>12,}  ({pct:.3f}%)   │")
print(f"│  Frozen (base model)       : {total - trainable:>12,}              │")
print("└─────────────────────────────────────────────────────────────┘")
print(f"\n📝 Use {pct:.3f}% in your report (replaces '~0.4%' estimate)")

del base, peft_model
torch.cuda.empty_cache()

---
## 📝 Section 6 — XL-Sum Summarization Experiments (Phase 1C)
Run these after the SQuAD experiments are complete.

In [ ]:
# ── 6A  Base SLM summarization baseline ───────────────────────────────────────

print("Running Base SLM zero-shot summarization baseline on XL-Sum...")
print("Expected time: ~15-20 min on T4\n")

!python src/training/run_pilot.py \
    --config configs/pilot_xlsum_summarization.yaml \
    --base_only

print("\n✅ Step 6A complete")

In [ ]:
# ── 6B  LoRA fine-tuning on XL-Sum (~2 hours) ─────────────────────────────────

print("Training SLM + LoRA on XL-Sum summarization...")
print("Expected time: ~2 hours on T4 (3 epochs × 5000 examples)\n")

!python src/training/run_pilot.py \
    --config configs/pilot_xlsum_summarization.yaml

print("\n✅ Step 6B complete")

In [ ]:
# ── 6C  LLM summarization baseline ────────────────────────────────────────────
# Note: you may need configs/llm_baseline_xlsum.yaml if the QA config can't be reused.
# Check that your llm config has task: summarization.

!python src/training/run_pilot.py \
    --config configs/llm_baseline_qwen7b.yaml \
    --base_only

print("\n✅ Step 6C complete")

---
## 📈 Section 7 — View All Results & Plots

In [ ]:
# ── 7.1  Full results table ────────────────────────────────────────────────────

import pandas as pd

try:
    df = pd.read_csv("experiments/results.csv")
    print(f"Total logged runs: {len(df)}\n")
    display(df)
except FileNotFoundError:
    print("No results.csv yet — run the experiment cells first")

In [ ]:
# ── 7.2  QA comparison bar chart ──────────────────────────────────────────────

import pandas as pd
import matplotlib.pyplot as plt

try:
    df = pd.read_csv("experiments/results.csv")
    qa_df = df[df["task"] == "qa"].copy()

    if len(qa_df) == 0:
        print("No QA results yet — run Section 5 first")
    else:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        fig.suptitle("SQuAD v2 QA — 3-Way Comparison", fontsize=14, fontweight="bold")

        labels  = qa_df["method"].tolist()
        em_vals = qa_df["exact_match"].tolist()
        f1_vals = qa_df["f1"].tolist()
        colors  = ["#6b7280", "#3b82f6", "#8b5cf6"]

        axes[0].bar(labels, em_vals, color=colors[:len(labels)])
        axes[0].set_title("Exact Match (%)"); axes[0].set_ylabel("%")
        axes[0].tick_params(axis="x", rotation=20)

        axes[1].bar(labels, f1_vals, color=colors[:len(labels)])
        axes[1].set_title("F1 Score (%)"); axes[1].set_ylabel("%")
        axes[1].tick_params(axis="x", rotation=20)

        plt.tight_layout()
        plt.savefig("experiments/qa_comparison.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("\n✅ Chart saved to experiments/qa_comparison.png")

except FileNotFoundError:
    print("No results.csv yet")

In [ ]:
# ── 7.3  Summarization (ROUGE) comparison bar chart ───────────────────────────

import pandas as pd
import matplotlib.pyplot as plt

try:
    df = pd.read_csv("experiments/results.csv")
    summ_df = df[df["task"] == "summarization"].copy()

    if len(summ_df) == 0:
        print("No summarization results yet — run Section 6 first")
    else:
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        fig.suptitle("XL-Sum Summarization — 3-Way Comparison", fontsize=14, fontweight="bold")

        labels = summ_df["method"].tolist()
        colors = ["#6b7280", "#3b82f6", "#8b5cf6"]

        for ax, metric, title in zip(
            axes,
            ["rouge1", "rouge2", "rougeL"],
            ["ROUGE-1", "ROUGE-2", "ROUGE-L"]
        ):
            vals = summ_df[metric].tolist()
            ax.bar(labels, vals, color=colors[:len(labels)])
            ax.set_title(title); ax.set_ylabel("Score")
            ax.tick_params(axis="x", rotation=20)

        plt.tight_layout()
        plt.savefig("experiments/rouge_comparison.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("\n✅ Chart saved to experiments/rouge_comparison.png")

except FileNotFoundError:
    print("No results.csv yet")

In [ ]:
# ── 7.4  Full experiment summary ───────────────────────────────────────────────

from src.utils.experiment_tracker import ExperimentTracker
ExperimentTracker("experiments/results.csv").print_summary()

---
## 💾 Section 8 — Save Results Back to Google Drive
Run this at the end of each session to save your checkpoints and results.

In [ ]:
# ── 8.1  Copy experiments folder back to Google Drive ─────────────────────────
# Only needed if your project is NOT already in Google Drive.

import shutil, os

SAVE_TO_DRIVE = False   # ← Set True to enable
DRIVE_SAVE_PATH = "/content/drive/MyDrive/Mtech_results"   # ← change as needed

if SAVE_TO_DRIVE:
    os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)
    shutil.copytree("experiments", f"{DRIVE_SAVE_PATH}/experiments", dirs_exist_ok=True)
    print(f"✅ Saved experiments/ to {DRIVE_SAVE_PATH}")
else:
    print("ℹ️  SAVE_TO_DRIVE is False — set it True to backup your results")

In [ ]:
# ── 8.2  Download results.csv directly to your computer ───────────────────────

from google.colab import files
import os

if os.path.exists("experiments/results.csv"):
    files.download("experiments/results.csv")
    print("✅ Downloading results.csv")
else:
    print("❌ results.csv not found — run experiments first")

---
## 💡 Tips & Troubleshooting

| Problem | Solution |
|---|---|
| **No GPU / CUDA not available** | Runtime → Change runtime type → T4 GPU |
| **OOM (out of memory) during training** | Set `load_in_bits: 4` in the YAML config (QLoRA) |
| **OOM during 7B inference** | Set `load_in_bits: 4` in `configs/llm_baseline_qwen7b.yaml` |
| **Session disconnects after 12 hours** | Save to Google Drive (Section 8) before session ends |
| **Import error after install** | Restart runtime: Runtime → Restart runtime, then re-run from Section 0.3 |
| **smoke_test [FAIL]** | Read the error message carefully — it tells you exactly what failed |
| **Loss not decreasing** | Check learning rate in YAML config; try reducing batch size |
| **`data/processed/` missing** | Run Section 4 data preparation cells |

### Key file locations after training
```
experiments/
├── results.csv                    ← all logged metrics (open this in Excel)
├── squad_qa/
│   ├── adapter/                   ← saved LoRA adapter weights
│   └── test_indices.json          ← which test examples were used
└── xlsum_summarization/
    └── adapter/                   ← XL-Sum LoRA adapter
```